# Build super-resolved real crops

Every real crop on the training side of `lab_ffpp_real` (DeepFakeFace and FaceForensics++ originals) is shrunk `FACTOR` times and enlarged back by a super-resolution network, then stored as a bona fide crop of the same person and pair group (`sources/superresolved.py`). `network_for` gives half the crops to Real-ESRGAN x4plus ([xinntao/Real-ESRGAN](https://github.com/xinntao/Real-ESRGAN), BSD-3-Clause), loaded with [spandrel](https://github.com/chaiNNer-org/spandrel) (MIT), and half to Swin2SR real-world x4 ([caidas/swin2SR-realworld-sr-x4-64-bsrgan-psnr](https://huggingface.co/caidas/swin2SR-realworld-sr-x4-64-bsrgan-psnr), Apache-2.0).

The `lab_ffpp_real_sr` protocol trains on these copies next to `lab_ffpp_real`'s training side, to test whether the lab's photo detector stops flagging real photos that a super-resolution network touched, as it flags DiffusionFace's CelebA-HQ photos (`docs/results.md`, 2026-10-19). Copies of calibration or test photos are built too but never enter training.

**Runtime:** Colab Pro, L4 or A100. **Needs** the Colab secret `HF_TOKEN` with write access.

### Configuration

In [ ]:
REPO_SOURCE = "git+https://github.com/paulohenriquevn/face-decision-lab@workspace"
HF_DATASET = "paulohenriquevn/face-decision-data"
FFPP_DATASET = "paulohenriquevn/face-decision-ffpp"
OUTPUT_DATASET = "paulohenriquevn/face-decision-superresolved"  # private
BUILD_DIR = "/content/build_superresolved"
REALESRGAN_WEIGHTS = "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth"
SWIN2SR_MODEL = "caidas/swin2SR-realworld-sr-x4-64-bsrgan-psnr"
BATCH_SIZE = 64

In [ ]:
%%capture
!pip install -q "face-decision-lab[build] @ {REPO_SOURCE}" spandrel

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token = userdata.get("HF_TOKEN"))

### Real crops on the training side

Only the training side gets copies in `lab_ffpp_real_sr`, so only it is built.

In [ ]:
from face_decision_lab.hub import load_hub_samples
from face_decision_lab.protocol import protocol_split

lab, lab_images = load_hub_samples(HF_DATASET)
ffpp, ffpp_images = load_hub_samples(FFPP_DATASET, key_prefix = "faceforensics")
images = dict(zip([s.image_path for s in lab + ffpp], lab_images + ffpp_images, strict = True))
train, _, _ = protocol_split(lab + ffpp, "lab_ffpp_real")
reals = [s for s in train if s.is_bona_fide]
print(len(reals), "real crops by source:", {d: sum(s.source_dataset == d for s in reals) for d in sorted({s.source_dataset for s in reals})})

### Networks

In [ ]:
import urllib.request
import torch
from spandrel import ModelLoader
from transformers import Swin2SRForImageSuperResolution

urllib.request.urlretrieve(REALESRGAN_WEIGHTS, "/content/RealESRGAN_x4plus.pth")
realesrgan = ModelLoader().load_from_file("/content/RealESRGAN_x4plus.pth").cuda().eval()
swin2sr = Swin2SRForImageSuperResolution.from_pretrained(SWIN2SR_MODEL).cuda().eval()

# Both take RGB in [0, 1] and enlarge 4x; Swin2SR needs sides that are multiples of its window (8).
def enlarge(network, pixels):
    if network == "realesrgan_x4plus":
        return realesrgan(pixels)
    return swin2sr(pixel_values = pixels).reconstruction

### Build

In [ ]:
import json
from pathlib import Path
import cv2
import numpy as np
from tqdm.auto import tqdm
from face_decision_lab.faces import CROP_SIZE, encode_jpeg
from face_decision_lab.manifest import write_manifest
from face_decision_lab.rows import open_image
from face_decision_lab.sources import superresolved

side = CROP_SIZE // superresolved.FACTOR
assert side % 8 == 0, side
output = Path(BUILD_DIR)
(output / "images" / superresolved.SOURCE_DATASET).mkdir(parents = True, exist_ok = True)
copies, counts, originals = [], {}, {}
for network in superresolved.NETWORKS:
    chosen = [s for s in reals if superresolved.network_for(s) == network]
    counts[network] = len(chosen)
    for start in tqdm(range(0, len(chosen), BATCH_SIZE), desc = network):
        batch = chosen[start:start + BATCH_SIZE]
        small = np.stack([cv2.resize(np.asarray(open_image(images[s.image_path])), (side, side), interpolation = cv2.INTER_AREA) for s in batch])
        with torch.no_grad():
            pixels = torch.from_numpy(small).permute(0, 3, 1, 2).float().div(255).cuda()
            large = enlarge(network, pixels)[:, :, :CROP_SIZE, :CROP_SIZE].clamp(0, 1)
        assert large.shape[-2:] == (CROP_SIZE, CROP_SIZE), large.shape
        for offset, (sample, crop) in enumerate(zip(batch, large.mul(255).round().byte().permute(0, 2, 3, 1).cpu().numpy(), strict = True)):
            relative = f"images/{superresolved.SOURCE_DATASET}/{network}_{start + offset:06d}.jpg"
            (output / relative).write_bytes(encode_jpeg(cv2.cvtColor(crop, cv2.COLOR_RGB2BGR)))
            copies.append(superresolved.super_resolved(sample, network, relative))
            originals[relative] = sample
write_manifest(copies, output / "manifest.csv")
(output / "report.json").write_text(json.dumps({"copies_by_network": counts, "factor": superresolved.FACTOR}, indent = 2))
print(len(copies), "copies", counts)

### Check

Sharpness (median variance of the Laplacian) of the originals and of each network's copies; the diagnosis measured 441 for DiffusionFace's CelebA-HQ crops and 278 for DeepFakeFace's. The grid shows originals over their copies.

In [ ]:
from PIL import Image

def sharpness(image):
    return float(cv2.Laplacian(cv2.cvtColor(np.asarray(image), cv2.COLOR_RGB2GRAY), cv2.CV_64F).var())

for network in superresolved.NETWORKS:
    some = [c for c in copies if c.generator == f"sr_{network}"][:500]
    print(f"{network:22s} copies {np.median([sharpness(Image.open(output / c.image_path)) for c in some]):7.1f}"
          f"  originals {np.median([sharpness(open_image(images[originals[c.image_path].image_path])) for c in some]):7.1f}")
row = [c for n in superresolved.NETWORKS for c in copies if c.generator == f"sr_{n}"][::max(1, len(copies) // 8)][:8]
grid = Image.new("RGB", (CROP_SIZE * len(row), CROP_SIZE * 2))
for i, c in enumerate(row):
    grid.paste(open_image(images[originals[c.image_path].image_path]), (CROP_SIZE * i, 0))
    grid.paste(Image.open(output / c.image_path), (CROP_SIZE * i, CROP_SIZE))
grid.resize((grid.width // 2, grid.height // 2))

### Upload

In [ ]:
from face_decision_lab.hub import push_build

print(push_build(BUILD_DIR, OUTPUT_DATASET))